<h1>🕸️ Biofilter — Report: <code>expand_entity_neighborhood</code></h1>

What sits one hop from each of these entities.

Takes a **heterogeneous** list — genes, diseases, proteins, GO terms, in
any mix — and reports what each one is connected to: how many neighbours,
of which kinds, and which ones.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "expand_entity_neighborhood"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

# Results land here whatever directory the kernel was started in — VS Code
# and Jupyter disagree about that, and a bare filename ends up wherever
# they landed. The project root is the folder holding .biofilter.toml.
_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

Mix whatever you like. A `gene:` style prefix narrows the search to that
group; without one, every group is searched.

In [ ]:
items = [
    "gene:TP53",              # restricted to Genes
    "protein:P04637",         # restricted to Proteins
    "MONDO:0007254",          # a CURIE — searched everywhere
    "APOE",                   # a bare name
    "ZZZ_NOT_A_THING",        # kept, with status='not_found'
]

result = bf.report.run(REPORT, input_data=items)
df = result.to_pandas()
df[["input_value", "input_type_hint", "entity_id", "entity_group",
    "degree_total", "status"]]

### 4. Why `GO:0006915` is not a hint

A prefix counts as a type hint **only when it names an entity group** —
and the group names are read from the bundle, not hardcoded.

That rule exists because of a defect in the report this replaces. It
treated *any* prefix before a colon as a hint, so `GO:0006915` became the
term `0006915`, which resolved to a **disease**. Every CURIE in the
bundle was affected — `MONDO:`, `HGNC:`, `DOID:`, `EFO:` — and nothing
failed loudly.

`go:` is excluded from the hint vocabulary for the same reason. Use
`go_terms:` to restrict to that group.

In [ ]:
bf.report.run(REPORT, input_data=[
    "GO:0006915",              # the GO term itself
    "MONDO:0007254",           # the disease itself
    "go_terms:GO:0006915",     # the same term, restricted explicitly
]).to_pandas()[["input_value", "input_type_hint", "entity_id", "entity_group", "status"]]

### 5. A hint narrows, and can rule out

`TP53` is a gene. Asking for it as a disease is a question with the
answer "no".

In [ ]:
bf.report.run(REPORT, input_data=["gene:TP53", "disease:TP53"]).to_pandas()[
    ["input_value", "entity_group", "status", "note"]
]

### 6. The neighbourhood itself

In [ ]:
for _, row in df[df["status"] == "ok"].iterrows():
    print(f"{row['input_value']}  ->  {row['primary_name']}  "
          f"({row['entity_group']}, degree {row['degree_total']:,})")
    for entry in row["neighbors_by_type"]:
        print(f"    {entry['group_name']:<14} {entry['count']:>6}  "
              f"{list(entry['names'])[:3]}")
    print()

`neighbors_by_type` is **one nested column**, not one column per entity
group. The report this replaces added a column per group present in the
bundle — 29 columns, 14 of them impossible to know before running it,
each holding a JSON string.

The names are capped by `neighbors_top_n_per_type`; `count` and
`degree_total` never are.

In [ ]:
capped = bf.report.run(REPORT, input_data=["gene:TP53"],
                       neighbors_top_n_per_type=3).to_pandas()

entry = capped.iloc[0]["neighbors_by_type"][0]
print(f"{entry['group_name']}: count={entry['count']:,}, names shown={len(entry['names'])}")
print(list(entry["names"]))

### 7. Degree zero is an answer

An entity can resolve cleanly and have nothing linked to it. GO terms are
the usual case: this bundle carries no entity relationships for them at
all, so the neighbourhood comes back empty and the `note` says why.

In [ ]:
bf.report.run(REPORT, input_data=["GO:0006915"]).to_pandas()[
    ["input_value", "entity_group", "degree_total", "status", "note"]
]

### 8. Export

CSV by default, with a `.provenance.json` beside it. The nested column is
written as JSON there; parquet keeps it as a real list.

In [ ]:
for path in result.write(OUTPUT_DIR / "expand_entity_neighborhood.csv"):
    print(path)

### 9. The same thing on the command line

```bash
biofilter report run --report-name expand_entity_neighborhood \\
    --input gene:BRCA1 --input "disease:breast cancer" \\
    --param neighbors_top_n_per_type=10 \\
    --output neighbourhood.csv
```

### 10. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("not found:", int((df["status"] == "not_found").sum()))
print("resolved but isolated:",
      int(((df["status"] == "ok") & (df["degree_total"] == 0)).sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))